# 第六节 从感知机到卷积神经网络

## 实验目标
通过本案例的学习：

1. 掌握更换模型网络结构的方法；
2. 掌握卷积神经网络的构建方法；


## 注意事项

1. 本案例推荐使用Pytorch-1.0.0，需使用 <font color='red' >GPU</font> 运行，请查看[《ModelArts CodeLab介绍》](https://support.huaweicloud.com/devtool-modelarts/devtool-modelarts_0010.html#section3)了解切换硬件规格的方法；

2. 如果您是第一次使用 JupyterLab，请查看[《ModelArts JupyterLab使用指导》](https://support.huaweicloud.com/devtool-modelarts/devtool-modelarts_0012.html)了解使用方法；

3. 如果您在使用 JupyterLab 过程中碰到报错，请参考[《ModelArts JupyterLab常见问题解决办法》](https://support.huaweicloud.com/modelarts_faq/modelarts_05_0185.html)尝试解决问题。

## 实验步骤

## 案例内容介绍
上一节我们使用十个输出节点的感知机模型实现了手写数字识别，但是在训练了3000个epoch之后，也仅仅达到0.8383的准确率，如果你尝试调整max_epochs、损失函数、梯度下降方法或学习率，你会发现准确率还是难以上去。用一句俗话来说，就是底层逻辑不变，只做一些表面功夫，始终难有较大的提升。
这种情况下，你或许就可以考虑更换模型的底层逻辑——网络结构了。接下来的几个案例，我们将分别使用CNN、LeNet-5和ResNet来实现手写数字识别，看它们的效果如何。本案例将使用CNN来实现。

### 1. 加载数据集
复用上一节保存的load_data_all函数，加载全量的手写数字识别数据集

In [1]:
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '../datasets/MNIST_data'))
from load_data_all import load_data_all

datasets_dir = '../datasets'
train_x, train_y, test_x, test_y = load_data_all(datasets_dir)

INFO:root:Using MoXing-v1.17.3-

INFO:root:Using OBS-Python-SDK-3.20.7


训练集规模： 60000 ，测试集规模： 10000


数据预处理  
本案例将使用卷积神经网络，由于卷积网络的输入要求是四维数组，第一维表示样本数量，第二维表示图像的通道数，第三四维表示图像的尺寸，所以我们要执行如下的代码，将train_x和test_x进行维度的转换

In [2]:
print("原维度：", train_x.shape, test_x.shape)
train_x = train_x.view(-1, 1, 28, 28)
test_x = test_x.view(-1, 1, 28, 28)
print("转换后的维度：", train_x.shape, test_x.shape)

原维度： torch.Size([60000, 784]) torch.Size([10000, 784])

转换后的维度： torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


### 2. 构建CNN网络和评价函数
评价函数直接复用了上一节的代码，网络结构部分就需要做较大的调整，代码如下，具体含义请查看代码注释

In [3]:
import torch
from torch import nn

class Network(nn.Module):
    def __init__(self, num_of_weights):
        """
        该网络只有三层网络，分别是卷积层1、卷积层2和全连接层1，ReLU和MaxPool2d由于不带参数，所以不计入网络层数
        """
        torch.manual_seed(0)
        super().__init__()
        
        # Convolution 1
        self.cnn1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=5, stride=1, padding=0)  # 卷积层1，输入为1个通道，输出为16个通道，卷积核大小为5，滑动步长为1，不做边缘填充
        self.relu1 = nn.ReLU()  # 激活层1，使用卷积网络中最常用的ReLU激活函数
        self.maxpool1 = nn.MaxPool2d(kernel_size=2)  # 最大池化层1
     
        # Convolution 2
        self.cnn2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=0)  # 卷积层2，输入为16个通道，输出为32个通道，卷积核大小为5，滑动步长为1，不做边缘填充
        self.relu2 = nn.ReLU()  # 激活层2，使用卷积网络中最常用的ReLU激活函数
        self.maxpool2 = nn.MaxPool2d(kernel_size=2)  # 最大池化层2
        
        # Fully connected 1
        self.fc1 = nn.Linear(32 * 4 * 4, 10)  # 全连接层1，输入维度为32*4*4，输出维度为10
    
    def forward(self, x):
        """
        前向传播函数
        """
        # Convolution 1
        out = self.cnn1(x)  # 卷积
        out = self.relu1(out)  # 激活
        out = self.maxpool1(out)  # 池化
        
        # Convolution 2 
        out = self.cnn2(out)  # 卷积
        out = self.relu2(out)  # 激活
        out = self.maxpool2(out)  # 池化
        
        # Fully connected 1
        out = out.view(out.size(0), -1)  # 输入到全连接层之前需要将32个4*4大小的特性矩阵拉成一个一维向量
        out = self.fc1(out)  # 计算全连接层
        
        return out
        
    def evaluate(self, pred_y, true_y):
        """
        准确率统计函数，与上一节一致
        """
        pred_labels = torch.argmax(pred_y, dim=1)
        acc = (pred_labels == true_y).float().mean()
        return acc

### 3. 交叉熵损失函数
与上一节代码一致

In [4]:
import torch.nn.functional as F
loss_fun = F.cross_entropy

### 4. 实现GPU训练的梯度下降算法
梯度下降算法的实现与上一节相比，就多了net.to(device)这一行，这一行的含义是：如果当前机器的cuda库可用，则表示有GPU，用GPU进行模型的训练，如果不可用，则用CPU进行训练。本案例的CNN网络只有三层，虽然用CPU也可以训练，但训练较慢，建议用GPU进行训练，切换硬件规格的方法，请查看本案例开头的注意事项

In [5]:
net = Network(28*28)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = net.to(device)
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

### 5. 实现训练函数
与上一节代码相比，仅增加了下面两行代码：  
train_x, train_y = train_x.to(device), train_y.to(device)  
test_x, test_y = test_x.to(device), test_y.to(device)

含义是：如果是要在GPU上训练，则将数据转换成cuda格式，拷贝到GPU上进行训练。

In [6]:
def train(net, train_x, train_y, test_x, test_y, max_epochs=100):
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    for epoch in range(1, max_epochs + 1):
        net.train()  # 切换为训练模式
        train_x, train_y = train_x.to(device), train_y.to(device)
        pred_y_train = net.forward(train_x)  # 前向传播
        train_loss = loss_fun(pred_y_train, train_y)  # 计算损失

        # 计算梯度，更新权值
        train_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (epoch == 1) or (epoch % 200 == 0):
            net.eval()  # 切换为评价模式，评价模式不计算梯度，计算更快
            test_x, test_y = test_x.to(device), test_y.to(device)
            pred_y_test = net.forward(test_x)
            test_loss = loss_fun(pred_y_test, test_y)
            train_acc = net.evaluate(pred_y_train, train_y)
            test_acc = net.evaluate(pred_y_test, test_y)
            print('epoch %d, train_loss %.4f, test_loss %.4f, train_acc: %.4f, test_acc: %.4f' % (epoch, train_loss.item(), test_loss.item(), train_acc, test_acc))
    return train_losses, test_losses, train_accs, test_accs

### 6. 开始训练
代码与上一节一致  
训练耗时约13分钟

In [7]:
import time
start_time = time.time()
max_epochs = 3000
train_losses, test_losses, train_accs, test_accs = train(net, train_x, train_y, test_x, test_y, max_epochs=max_epochs)
print('cost time: %.1f s' % (time.time() - start_time))

epoch 1, train_loss 2.3208, test_loss 2.3210, train_acc: 0.1044, test_acc: 0.1062

epoch 200, train_loss 1.2535, test_loss 1.2217, train_acc: 0.7570, test_acc: 0.7715

epoch 400, train_loss 0.4686, test_loss 0.4412, train_acc: 0.8693, test_acc: 0.8768

epoch 600, train_loss 0.3620, test_loss 0.3388, train_acc: 0.8947, test_acc: 0.9015

epoch 800, train_loss 0.3092, test_loss 0.2878, train_acc: 0.9094, test_acc: 0.9163

epoch 1000, train_loss 0.2718, test_loss 0.2518, train_acc: 0.9196, test_acc: 0.9272

epoch 1200, train_loss 0.2426, test_loss 0.2238, train_acc: 0.9281, test_acc: 0.9375

epoch 1400, train_loss 0.2187, test_loss 0.2010, train_acc: 0.9350, test_acc: 0.9426

epoch 1600, train_loss 0.1989, test_loss 0.1822, train_acc: 0.9413, test_acc: 0.9493

epoch 1800, train_loss 0.1823, test_loss 0.1665, train_acc: 0.9460, test_acc: 0.9527

epoch 2000, train_loss 0.1684, test_loss 0.1534, train_acc: 0.9502, test_acc: 0.9564

epoch 2200, train_loss 0.1568, test_loss 0.1424, train_acc: 0

从上面的输出，可以看到CNN网络在训练3000个epoch之后，达到了0.9673的准确率

## 本节扩展学习材料
[卷积神经网络](https://education.huaweicloud.com/courses/course-v1:HuaweiX+CBUCNXE088+Self-paced/courseware/2aaefc7acffd43ab8eff5cacce98998b/8b8f8068bfa64d8bb6c4fefb98d46a6b/)

至此，本案例完成。